## 1. Data Preparation
Reads and cleans the raw CSV, splits columns across 3 source systems

Claude helped me split up the CSV file into the 3 formats (MongoDB, MySQL, and a CSV file) due to me using a new dataset and not adventure works.

#### Import Libraries

In [1]:
import os
import json
import numpy
import datetime
import certifi
import pymongo

import pandas as pd

import sqlalchemy
from sqlalchemy import create_engine, text

#### Declare Connection Variables

In [2]:
mysql_src_args = {
    "uid"      : "root",
    "pwd"      : "Frthy1!234",  
    "hostname" : "localhost",
    "dbname"   : "hotel_src"
}

mysql_args = {
    "uid"      : "root",
    "pwd"      : "Frthy1!234",  
    "hostname" : "localhost",
    "dbname"   : "hotel_dw"
}

mongodb_args = {
    "user_name" : "",
    "password" : "",
    "cluster_name" : "",
    "cluster_subnet" : "",
    "cluster_location" : "local", 
    "db_name" : "hotel_bookings"
}


data_dir         = os.path.join(os.getcwd(), 'data')
raw_csv          = os.path.join(data_dir, 'hotel_bookings.csv')
clean_bookings   = os.path.join(data_dir, 'bookings_clean.csv')
clean_room_types = os.path.join(data_dir, 'room_types_clean.csv')

#### Define Helper Functions

In [3]:
def get_sql_dataframe(sql_query, **args):
    '''Create a connection to the MySQL database'''
    conn_str = f"mysql+pymysql://{args['uid']}:{args['pwd']}@{args['hostname']}/{args['dbname']}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    connection = sqlEngine.connect()

    '''Invoke the pd.read_sql() function to query the database, and fill a Pandas DataFrame.'''
    dframe = pd.read_sql(text(sql_query), connection);
    connection.close()

    return dframe


def set_dataframe(df, table_name, pk_column, db_operation, **args):
    '''Create a connection to the MySQL database'''
    conn_str = f"mysql+pymysql://{args['uid']}:{args['pwd']}@{args['hostname']}/{args['dbname']}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    db_connection = sqlEngine.connect()

    '''Invoke the Pandas DataFrame .to_sql( ) function to either create, or append to, a table'''
    if db_operation in ['insert', 'update']:
        if db_operation.lower() == "insert":
            df.to_sql(table_name, con=db_connection, index=False, if_exists='replace')
            db_connection.execute(text(f"ALTER TABLE {table_name} ADD {pk_column} INT AUTO_INCREMENT PRIMARY KEY FIRST;"))

        elif db_operation.lower() == "update":
            df.to_sql(table_name, con=db_connection, index=False, if_exists='append')
    else:
        print("The value supplied to the 'db_operation' parameter must be either 'insert' or 'update'.")

    db_connection.close()


def get_mongo_client(**args):
    '''Validate proper input'''
    if args["cluster_location"] not in ['atlas', 'local']:
        raise Exception("You must specify either 'atlas' or 'local' for the cluster_location parameter.")
    else:
        if args["cluster_location"] == "atlas":
            import certifi
            connect_str  = f"mongodb+srv://{args['user_name']}:{args['password']}@"
            connect_str += f"{args['cluster_name']}.{args['cluster_subnet']}.mongodb.net"
            client = pymongo.MongoClient(connect_str, tlsCAFile=certifi.where())
        elif args["cluster_location"] == "local":
            client = pymongo.MongoClient("mongodb://localhost:27017/")

    return client


def get_mongo_dataframe(mongo_client, db_name, collection, query):
    '''Query MongoDB, and fill a python list with documents to create a DataFrame'''
    db = mongo_client[db_name]
    dframe = pd.DataFrame(list(db[collection].find(query)))
    dframe.drop(['_id'], axis=1, inplace=True)
    mongo_client.close()

    return dframe


def set_mongo_collection(mongo_client, db_name, collection_name, records):
    '''Drop and re-insert a MongoDB collection — idempotent'''
    db = mongo_client[db_name]
    db.drop_collection(collection_name)
    db[collection_name].insert_many(records)

#### Create Destination Data Warehouse

In [4]:
conn_str   = f"mysql+pymysql://{mysql_args['uid']}:{mysql_args['pwd']}@{mysql_args['hostname']}"
sqlEngine  = create_engine(conn_str, pool_recycle=3600)
connection = sqlEngine.connect()

connection.execute(text(f"DROP DATABASE IF EXISTS `{mysql_args['dbname']}` ;"))
connection.execute(text(f"CREATE DATABASE `{mysql_args['dbname']}` ;"))
connection.close()

---
#### Read & Clean the Raw CSV

In [5]:
df_raw = pd.read_csv(raw_csv)
df_raw.head(2)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01


#### Clean Data

In [6]:
# Fix null values (in columns with few nulls)
df_raw['children'] = df_raw['children'].fillna(0).astype(int)
df_raw['country'] = df_raw['country'].fillna('Unknown')

# Drop cols with lots of null values or cols that are not useful to us
drop_cols = [
    'agent', 'company', 'arrival_date_week_number',
    'reservation_status', 'reservation_status_date',
    'previous_cancellations', 'previous_bookings_not_canceled'
]
df_raw.drop(columns=drop_cols, inplace=True)

# convert arrival_date to proper datetime format
month_map = {
    'January':1,  'February':2,  'March':3,     'April':4,
    'May':5,      'June':6,      'July':7,       'August':8,
    'September':9,'October':10,  'November':11,  'December':12
}

df_raw['arrival_date'] = pd.to_datetime({
    'year'  : df_raw['arrival_date_year'],
    'month' : df_raw['arrival_date_month'].map(month_map),
    'day'   : df_raw['arrival_date_day_of_month']
}).dt.date

df_raw.drop(columns=['arrival_date_year', 'arrival_date_month',
                      'arrival_date_day_of_month'], inplace=True)

print(df_raw.shape)

(119390, 23)


#### Sample Down to 3000 rows for a more managable dataset (original was ~120k as shown above)

In [7]:
df_resort = df_raw[df_raw['hotel'] == 'Resort Hotel'].sample(frac=0.025, random_state=42)
df_city   = df_raw[df_raw['hotel'] == 'City Hotel'].sample(frac=0.025, random_state=42)
df_raw    = pd.concat([df_resort, df_city]).reset_index(drop=True)

print(df_raw.shape)

(2985, 23)


### Split into 3 sources

#### MySQL data (guest data)

In [8]:
guest_cols = ['country', 'customer_type', 'adults', 'children', 'babies', 'is_repeated_guest', 'required_car_parking_spaces']

df_guests = df_raw[guest_cols].copy()
df_guests.insert(0, 'guest_id', range(1, len(df_guests) + 1))

df_guests.head(2)

,guest_id,country,customer_type,adults,children,babies,is_repeated_guest,required_car_parking_spaces
0,1,SWE,Transient,2,0,0,0,1
1,2,POL,Transient-Party,2,0,0,0,0


In [9]:
conn_str  = f"mysql+pymysql://{mysql_src_args['uid']}:{mysql_src_args['pwd']}@{mysql_src_args['hostname']}"
sqlEngine = create_engine(conn_str, pool_recycle=3600)
connection = sqlEngine.connect()
connection.execute(text(f"CREATE DATABASE IF NOT EXISTS `{mysql_src_args['dbname']}` ;"))
connection.close()

conn_str      = f"mysql+pymysql://{mysql_src_args['uid']}:{mysql_src_args['pwd']}@{mysql_src_args['hostname']}/{mysql_src_args['dbname']}"
sqlEngine     = create_engine(conn_str, pool_recycle=3600)
db_connection = sqlEngine.connect()
df_guests.to_sql('guests', con=db_connection, index=False, if_exists='replace')
db_connection.close()

df_guests.head(2)

,guest_id,country,customer_type,adults,children,babies,is_repeated_guest,required_car_parking_spaces
0,1,SWE,Transient,2,0,0,0,1
1,2,POL,Transient-Party,2,0,0,0,0


#### MongoDB data

In [10]:
hotel_cols = ['hotel', 'market_segment', 'distribution_channel', 'deposit_type']

df_hotels = df_raw[hotel_cols].copy()
df_hotels.insert(0, 'hotel_id', range(1, len(df_hotels) + 1))

client = get_mongo_client(**mongodb_args)
set_mongo_collection(client, mongodb_args['db_name'], 'hotels', df_hotels.to_dict(orient='records'))
client.close()

#### CSV data

In [11]:
room_cols = ['reserved_room_type', 'assigned_room_type', 'meal']

df_room_types = df_raw[room_cols].copy()
df_room_types.insert(0, 'room_id', range(1, len(df_room_types) + 1))

df_room_types.to_csv(clean_room_types, index=False)
df_room_types.head(2)

,room_id,reserved_room_type,assigned_room_type,meal
0,1,A,A,BB
1,2,A,C,HB


#### Other columns for fact table

In [16]:
keep_cols = [
    'guest_id', 'hotel_id', 'room_id', 'arrival_date',
    'lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights',
    'booking_changes', 'days_in_waiting_list',
    'total_of_special_requests', 'adr', 'is_canceled'
]

df_bookings = df_raw.copy()
df_bookings['guest_id']     = df_guests['guest_id'].values
df_bookings['hotel_id']     = df_hotels['hotel_id'].values
df_bookings['room_id'] = df_room_types['room_id'].values

df_bookings = df_bookings[keep_cols]
df_bookings.to_csv(clean_bookings, index=False)
df_bookings.head(2)

,guest_id,hotel_id,room_id,arrival_date,lead_time,stays_in_weekend_nights,stays_in_week_nights,booking_changes,days_in_waiting_list,total_of_special_requests,adr,is_canceled
0,1,1,1,2016-06-21,167,1,5,0,0,0,82.0,0
1,2,2,2,2016-06-22,322,0,3,1,75,0,92.0,0


### 2. ETL Pipeline

#### Create dim tables

In [17]:
# dim guests
sql_guests = "SELECT * FROM hotel_src.guests;"
df_guests = get_sql_dataframe(sql_guests, **mysql_src_args)
df_guests.head(2)

,guest_id,country,customer_type,adults,children,babies,is_repeated_guest,required_car_parking_spaces
0,1,SWE,Transient,2,0,0,0,1
1,2,POL,Transient-Party,2,0,0,0,0


In [18]:
# dim hotels
client = get_mongo_client(**mongodb_args)
query = {}
df_hotels = get_mongo_dataframe(client, mongodb_args['db_name'], 'hotels', query)
df_hotels.head(2)

,hotel_id,hotel,market_segment,distribution_channel,deposit_type
0,1,Resort Hotel,Online TA,TA/TO,No Deposit
1,2,Resort Hotel,Groups,Direct,No Deposit


In [19]:
# dim room_types
df_rooms = pd.read_csv(clean_room_types)
df_rooms.head(2)

,room_id,reserved_room_type,assigned_room_type,meal
0,1,A,A,BB
1,2,A,C,HB


#### Run dim_date.sql

#### Load DataFrames into the new Data Warehouse

In [20]:
dataframe = df_guests
table_name = 'dim_guests'
primary_key = 'guest_key'
db_operation = "insert"

set_dataframe(dataframe, table_name, primary_key, db_operation, **mysql_args)

In [21]:
dataframe = df_hotels
table_name = 'dim_hotels'
primary_key = 'hotel_key'
db_operation = "insert"

set_dataframe(dataframe, table_name, primary_key, db_operation, **mysql_args)

In [22]:
dataframe = df_rooms
table_name = 'dim_rooms'
primary_key = 'room_key'
db_operation = "insert"

set_dataframe(dataframe, table_name, primary_key, db_operation, **mysql_args)

#### Validate that the New Dimension Tables were Created

In [23]:
sql_guests = "SELECT * FROM hotel_dw.dim_guests;"
df_dim_guests = get_sql_dataframe(sql_guests, **mysql_args)
df_dim_guests.head(2)

,guest_key,guest_id,country,customer_type,adults,children,babies,is_repeated_guest,required_car_parking_spaces
0,1,1,SWE,Transient,2,0,0,0,1
1,2,86,PRT,Transient,2,0,0,0,0


In [24]:
sql_hotels = "SELECT * FROM hotel_dw.dim_hotels;"
df_dim_hotels = get_sql_dataframe(sql_hotels, **mysql_args)
df_dim_hotels.head(2)

,hotel_key,hotel_id,hotel,market_segment,distribution_channel,deposit_type
0,1,1,Resort Hotel,Online TA,TA/TO,No Deposit
1,2,2,Resort Hotel,Groups,Direct,No Deposit


In [25]:
sql_rooms = "SELECT * FROM hotel_dw.dim_rooms;"
df_dim_rooms = get_sql_dataframe(sql_rooms, **mysql_args)
df_dim_rooms.head(2)

,room_key,room_id,reserved_room_type,assigned_room_type,meal
0,1,1,A,A,BB
1,2,2,A,C,HB


### Create and Populate Fact Table

#### Extract Data from the Cleaned Bookings CSV we created earlier

In [26]:
df_bookings = pd.read_csv(clean_bookings)
df_bookings.head(2)

,guest_id,hotel_id,room_id,arrival_date,lead_time,stays_in_weekend_nights,stays_in_week_nights,booking_changes,days_in_waiting_list,total_of_special_requests,adr,is_canceled
0,1,1,1,2016-06-21,167,1,5,0,0,0,82.0,0
1,2,2,2,2016-06-22,322,0,3,1,75,0,92.0,0


#### Lookup the DateKeys from the Date Dimension Table

In [27]:
sql_dim_date = "SELECT date_key, full_date FROM hotel_dw.dim_date;"
df_dim_date = get_sql_dataframe(sql_dim_date, **mysql_args)
df_dim_date.full_date = df_dim_date.full_date.astype('datetime64[ns]').dt.date
df_dim_date.head(2)

,date_key,full_date
0,20140101,2014-01-01
1,20140102,2014-01-02


#### Lookup Surrogate Primary Key that Corresponds to arrival_date

In [28]:
df_dim_arrival = df_dim_date.rename(columns={"date_key" : "arrival_date_key", "full_date" : "arrival_date"})

df_bookings["arrival_date"] = pd.to_datetime(df_bookings["arrival_date"]).dt.date
df_bookings = pd.merge(df_bookings, df_dim_arrival, on="arrival_date", how="left")
df_bookings.drop(["arrival_date"], axis=1, inplace=True)
df_bookings.head(2)

,guest_id,hotel_id,room_id,lead_time,stays_in_weekend_nights,stays_in_week_nights,booking_changes,days_in_waiting_list,total_of_special_requests,adr,is_canceled,arrival_date_key
0,1,1,1,167,1,5,0,0,0,82.0,0,20160621
1,2,2,2,322,0,3,1,75,0,92.0,0,20160622


#### Lookup the Primary Keys from the Dimension Tables

In [29]:
df_dim_guest_keys = df_dim_guests[['guest_key', 'guest_id']]
df_bookings = pd.merge(df_bookings, df_dim_guest_keys, on="guest_id", how="left")
df_bookings.drop(["guest_id"], axis=1, inplace=True)
df_bookings.head(2)

,hotel_id,room_id,lead_time,stays_in_weekend_nights,stays_in_week_nights,booking_changes,days_in_waiting_list,total_of_special_requests,adr,is_canceled,arrival_date_key,guest_key,country,customer_type,adults,children,babies,is_repeated_guest,required_car_parking_spaces
0,1,1,167,1,5,0,0,0,82.0,0,20160621,1,SWE,Transient,2,0,0,0,1
1,2,2,322,0,3,1,75,0,92.0,0,20160622,3,POL,Transient-Party,2,0,0,0,0


In [30]:
df_dim_hotel_keys = df_dim_hotels[['hotel_key', 'hotel_id']]
df_bookings = pd.merge(df_bookings, df_dim_hotel_keys, on="hotel_id", how="left")
df_bookings.drop(["hotel_id"], axis=1, inplace=True)
df_bookings.head(2)

,room_id,lead_time,stays_in_weekend_nights,stays_in_week_nights,booking_changes,days_in_waiting_list,total_of_special_requests,adr,is_canceled,arrival_date_key,...,adults,children,babies,is_repeated_guest,required_car_parking_spaces,hotel_key,hotel,market_segment,distribution_channel,deposit_type
0,1,167,1,5,0,0,0,82.0,0,20160621,...,2,0,0,0,1,1,Resort Hotel,Online TA,TA/TO,No Deposit
1,2,322,0,3,1,75,0,92.0,0,20160622,...,2,0,0,0,0,2,Resort Hotel,Groups,Direct,No Deposit


In [32]:
df_dim_room_keys = df_dim_rooms[['room_key', 'room_id']]
df_bookings = pd.merge(df_bookings, df_dim_room_keys, on="room_id", how="left")
df_bookings.drop(["room_id"], axis=1, inplace=True)
df_bookings.head(2)

,lead_time,stays_in_weekend_nights,stays_in_week_nights,booking_changes,days_in_waiting_list,total_of_special_requests,adr,is_canceled,arrival_date_key,guest_key,...,required_car_parking_spaces,hotel_key,hotel,market_segment,distribution_channel,deposit_type,room_key,reserved_room_type,assigned_room_type,meal
0,167,1,5,0,0,0,82.0,0,20160621,1,...,1,1,Resort Hotel,Online TA,TA/TO,No Deposit,1,A,A,BB
1,322,0,3,1,75,0,92.0,0,20160622,3,...,0,2,Resort Hotel,Groups,Direct,No Deposit,2,A,C,HB


#### Additional Transformations to the DataFrames

In [33]:
column_order = [
    "arrival_date_key", "guest_key", "hotel_key", "room_key", "lead_time", "stays_in_weekend_nights", "stays_in_week_nights", 
    "booking_changes", "days_in_waiting_list", "total_of_special_requests", "adr", "is_canceled"
]

df_fact = df_bookings[column_order]
df_fact.head(2)

,arrival_date_key,guest_key,hotel_key,room_key,lead_time,stays_in_weekend_nights,stays_in_week_nights,booking_changes,days_in_waiting_list,total_of_special_requests,adr,is_canceled
0,20160621,1,1,1,167,1,5,0,0,0,82.0,0
1,20160622,3,2,2,322,0,3,1,75,0,92.0,0


#### Write fact table back into Database

In [34]:
table_name = "fact_bookings"
primary_key = "booking_key"
db_operation = "insert"

set_dataframe(df_fact, table_name, primary_key, db_operation, **mysql_args)

#### Validate

In [35]:
sql_fact = "SELECT * FROM hotel_dw.fact_bookings LIMIT 5;"
get_sql_dataframe(sql_fact, **mysql_args)

,booking_key,arrival_date_key,guest_key,hotel_key,room_key,lead_time,stays_in_weekend_nights,stays_in_week_nights,booking_changes,days_in_waiting_list,total_of_special_requests,adr,is_canceled
0,1,20160621,1,1,1,167,1,5,0,0,0,82.00,0
1,2,20160622,3,2,2,322,0,3,1,75,0,92.00,0
2,3,20170529,5,3,3,128,1,2,0,0,1,115.00,1
3,4,20161226,7,4,4,125,2,5,0,0,0,56.88,1
4,5,20160913,8,5,6,0,0,3,0,0,0,165.50,0


## SQL Queries

#### Total Revenue and Bookings by Hotel From 2015 to 2017

In [53]:
sql_1 = """
    SELECT 
        h.hotel,
        d.calendar_year,
        COUNT(f.booking_key) AS total_bookings,
        ROUND(SUM(f.adr), 2) AS total_revenue,
        ROUND(AVG(f.stays_in_week_nights), 2) AS avg_week_nights,
        ROUND(AVG(f.lead_time), 1) AS avg_lead_time_days
    FROM hotel_dw.fact_bookings AS f
    INNER JOIN hotel_dw.dim_guests AS g ON f.guest_key = g.guest_key
    INNER JOIN hotel_dw.dim_hotels AS h on f.hotel_key = h.hotel_key
    INNER JOIN hotel_dw.dim_date AS d ON f.arrival_date_key = d.date_key
    GROUP BY h.hotel, d.calendar_year
    ORDER BY calendar_year ASC;
"""

df_1 = get_sql_dataframe(sql_1, **mysql_args)
df_1

,hotel,calendar_year,total_bookings,total_revenue,avg_week_nights,avg_lead_time_days
0,Resort Hotel,2015,200,17070.19,3.21,84.7
1,City Hotel,2015,345,29140.71,2.01,114.3
2,Resort Hotel,2016,477,42272.80,3.04,95.4
3,City Hotel,2016,957,98058.45,2.14,107.6
4,Resort Hotel,2017,325,34461.94,3.25,98.7
5,City Hotel,2017,681,80925.02,2.31,113.2


#### Average Daily Rate by Room Type, Meal Plan and Quarter

In [60]:
sql_2 = """
    SELECT
        h.hotel,
        r.reserved_room_type,
        r.meal,
        d.calendar_quarter,
        d.calendar_year,
        COUNT(f.booking_key) AS total_bookings,
        ROUND(AVG(f.adr), 2) AS avg_daily_rate,
        SUM(f.total_of_special_requests) AS total_special_requests
    FROM hotel_dw.fact_bookings AS f
    INNER JOIN hotel_dw.dim_rooms AS r ON f.room_key = r.room_key
    INNER JOIN hotel_dw.dim_hotels AS h ON f.hotel_key = h.hotel_key
    INNER JOIN hotel_dw.dim_date AS d ON f.arrival_date_key = d.date_key
    GROUP BY h.hotel, r.reserved_room_type, r.meal, d.calendar_quarter, d.calendar_year
    ORDER BY avg_daily_rate DESC;
"""

df_2 = get_sql_dataframe(sql_2, **mysql_args)
df_2.head(10)

,hotel,reserved_room_type,meal,calendar_quarter,calendar_year,total_bookings,avg_daily_rate,total_special_requests
0,Resort Hotel,F,Undefined,3,2017,1,367.00,0.0
1,Resort Hotel,H,BB,3,2017,2,286.30,2.0
2,Resort Hotel,G,HB,3,2016,3,281.45,0.0
3,Resort Hotel,G,HB,3,2017,1,279.00,1.0
4,Resort Hotel,E,HB,3,2017,1,277.00,4.0
5,Resort Hotel,F,HB,3,2017,2,270.30,1.0
6,Resort Hotel,C,FB,3,2015,1,261.40,0.0
7,City Hotel,G,BB,3,2017,3,261.15,2.0
8,City Hotel,F,HB,2,2017,1,258.00,0.0
9,Resort Hotel,G,BB,3,2017,3,254.44,2.0
